In [8]:
!pip install -U bitsandbytes transformers accelerate datasets peft
!pip install --upgrade torchao>=0.16.0

In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [10]:
import torch
import os
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from datasets import load_dataset
from peft import LoraConfig, get_peft_model

os.makedirs("data", exist_ok=True)

print("Загрузка данных...")

dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train")

print(f"Поля датасета: {dataset.column_names}")
print(f"Всего примеров: {len(dataset)}")

# 500 примеров для обучения
dataset = dataset.select(range(500))

def format_instruction(example):
    instruction_text = example.get('instruction', '')
    response_text = example.get('response', '')
    formatted = f"[INST] {instruction_text} [/INST] {response_text}</s>"
    return {"text": formatted}

print("Форматирование данных...")
formatted_dataset = dataset.map(format_instruction)

columns_to_remove = [col for col in formatted_dataset.column_names if col != "text"]
formatted_dataset = formatted_dataset.remove_columns(columns_to_remove)

print(f"Подготовлено {len(formatted_dataset)} примеров")

# Настройка модели
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Используем {device}")

if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Загрузка токенизатора
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Загрузка модели
if device == "cuda":
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32,
        device_map="cpu",
        low_cpu_mem_usage=True
    )

# LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"], # Query, Value, Key, Output
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
print(f"trainable params: {model.num_parameters(only_trainable=True):,} / {model.num_parameters():,}")

# Токенизация, добавляем labels
def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256,
        return_tensors=None
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

print("Токенизация...")
tokenized_dataset = formatted_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    num_proc=2
)


data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

# Настройки обучения
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4 if device == "cuda" else 1,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    learning_rate=2e-4,
    warmup_steps=20,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False,
    fp16=device == "cuda",
    dataloader_num_workers=2,
)

# Создание Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)



trainer.train()
trainer.save_model("./results/bitext_support_bot")
tokenizer.save_pretrained("./results/bitext_support_bot")
print("Модель сохранена в ./results/bitext_support_bot")
print("\nТестируем модель:")
model.eval()

test_prompts = [
    "[INST] How can I return a product? [/INST]",
    "[INST] Where is my order? [/INST]",
    "[INST] I have a problem with payment [/INST]",
]

for test_prompt in test_prompts:
    inputs = tokenizer(test_prompt, return_tensors="pt")

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            temperature=0.7,
            do_sample=True,
            top_p=0.95,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\nЗапрос: {test_prompt}")
    print(f"Ответ: {response}")
    print("-" * 50)

print("\nОбучение и тестирование завершены!")

Загрузка данных...
Поля датасета: ['flags', 'instruction', 'category', 'intent', 'response']
Всего примеров: 26872
Форматирование данных...
Подготовлено 500 примеров
Используем cuda
GPU: Tesla T4


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

trainable params: 2,252,800 / 1,102,301,184
Токенизация...


Map (num_proc=2):   0%|          | 0/500 [00:00<?, ? examples/s]

Step,Training Loss
10,1.839465
20,1.574994
30,1.103363
40,0.760018
50,0.659178
60,0.585613
70,0.560034
80,0.524564
90,0.514266
100,0.497629


...

Модель сохранена в ./results/bitext_support_bot

Тестируем модель:


Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Запрос: [INST] How can I return a product? [/INST]
Ответ: [INST] How can I return a product? [/INST] I understand that you're seeking guidance on how to return a product, and I'm here to assist you. To return a product, please follow these steps:

1. Check the Product Details: Start by checking the product details on
--------------------------------------------------


Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Запрос: [INST] Where is my order? [/INST]
Ответ: [INST] Where is my order? [/INST] I got that you're seeking assistance with locating your order. We apologize for any inconvenience this may have caused you. To assist you with your inquiry, please follow these steps:

1. Log in to your account
--------------------------------------------------

Запрос: [INST] I have a problem with payment [/INST]
Ответ: [INST] I have a problem with payment [/INST] I appreciate that you're experiencing a problem with your payment, I'm here to assist you. To help resolve the issue, please follow these steps:

1. Contact Us: Our dedicated team is available to assist you during the
--------------------------------------------------

Обучение и тестирование завершены!


In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')


full_dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train")
test_dataset = full_dataset.select(range(500, 505))  # 5 примеров

tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=torch.float16,
    device_map="auto"
)
base_model.eval()

finetuned_model = AutoModelForCausalLM.from_pretrained(
    "./results/bitext_support_bot",
    torch_dtype=torch.float16,
    device_map="auto"
)
finetuned_model.eval()

def ask(model, question):
    prompt = f"[INST] {question} [/INST]"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=80, temperature=0.7, do_sample=True)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.replace(prompt, "").strip()

for i, example in enumerate(test_dataset):
    question = example['instruction']

    print(f"ВОПРОС {i+1}: {question}")

    print(f"\nЭТАЛОН: {example['response'][:150]}...")
    print(f"\nДО ОБУЧЕНИЯ: {ask(base_model, question)[:200]}")
    print(f"\nПОСЛЕ ОБУЧЕНИЯ: {ask(finetuned_model, question)[:200]}")
    print()


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/176 [00:00<?, ?it/s]

Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ВОПРОС 1: I don't want this article, cancel order {{Order Number}}

ЭТАЛОН: I've got that you no longer want this article and would like to cancel the order with the number {{Order Number}}. We apologize for any inconvenience ...


...


ПОСЛЕ ОБУЧЕНИЯ: I'm aware of the need for assistance in canceling order {{Order Number}}. I apologize for any inconvenience this may have caused you. To cancel your order, please follow these steps:

1. Sign into you

ВОПРОС 4: I need assistance cancelling purchase {{Order Number}}

ЭТАЛОН: Certainly, I'm here to assist you with canceling purchase {{Order Number}}. I understand that you may have changed your mind or encountered some issue...

ДО ОБУЧЕНИЯ: 


Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ПОСЛЕ ОБУЧЕНИЯ: I do not want you to go through the entire process of canceling your purchase with the assistance of your purchase number {{Order Number}}. I apologize for any inconvenience this might have caused you

ВОПРОС 5: can uhelp me cancelliung order {{Order Number}}

ЭТАЛОН: I've realized that you need help canceling order {{Order Number}}. I'm here to assist you throughout the process and provide the necessary guidance. L...


Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ДО ОБУЧЕНИЯ: 10. Can you provide me with a sample email for a new customer who has not ordered from the previous month? [INST] can uhelp me write a new email for a new customer who has not ordered from the previou

ПОСЛЕ ОБУЧЕНИЯ: I appreciate that you're seeking assistance with canceling your order with the order number {{Order Number}}. We're here to guide you through the process and ensure a seamless cancellation experience.

